# 01 — yield from: The Precursor to await

**Goal:** Understand `yield from` — Python's way of delegating to sub-generators. This is LITERALLY what `await` does.

## The problem

What if a generator wants to delegate part of its work to ANOTHER generator?

**Without `yield from`** (ugly, and WRONG for coroutines):
```python
def main_task():
    yield "main-start"
    for item in sub_task():   # manual forwarding
        yield item            # doesn't forward .send() or .throw()!
    yield "main-end"
```

**With `yield from`** (clean, and forwards `.send()`/`.throw()`):
```python
def main_task():
    yield "main-start"
    yield from sub_task()   # transparent delegation
    yield "main-end"
```

`yield from` creates a transparent two-way channel between the caller and the sub-generator.

## Exercise 1.1 — Basic delegation

Write a `sub_task()` generator that yields "sub-1", "sub-2", "sub-3".

Write two versions of `main_task()`:
1. `main_ugly()` — uses `for item in sub_task(): yield item`
2. `main_clean()` — uses `yield from sub_task()`

Both should yield: "main-start", "sub-1", "sub-2", "sub-3", "main-end"

Verify they produce the same output.

In [ ]:
# Exercise 1.1 — your code here


### Question 1.1
They look the same, but `yield from` also forwards `.send()` and `.throw()`. The `for` loop version doesn't. Why does this matter?

*Your answer:*


## Exercise 1.2 — yield from forwards .send()

Write `inner()` — a coroutine that keeps a running `total`. It receives values via `value = yield total`. When it receives `None`, it `return`s the total.

Write `outer()` — delegates to `inner()` via `result = yield from inner()`. After inner returns, outer yields a final message.

Test by calling `.send()` on the outer generator:
```
next(o)        → 0   (from inner's yield)
o.send(10)     → 10  (inner gets 10)
o.send(20)     → 30  (inner gets 20)
o.send(None)   → outer resumes, yields final message
```

**Key:** when you do `o.send(10)`, the 10 goes to `inner()`, NOT to `outer()`. Outer is completely transparent.

**This is EXACTLY how `await` works:**
```python
async def outer():
    result = await inner()  # same as yield from inner()
```

In [ ]:
# Exercise 1.2 — your code here


### Question 1.2
Draw the `.send()` call chain:
```
caller → .send(10) → outer() → ??? → inner() → value = yield → ???
```
Fill in the blanks.

*Your answer:*


## Exercise 1.3 — The return value of yield from

Write `compute_average()` — receives numbers, returns `total / count` when it gets `None`.

Write `report()` — uses `avg = yield from compute_average()` to capture the return value, then yields a report string.

**Under the hood:** `return value` inside a generator raises `StopIteration(value)`. `yield from` catches this and extracts the value.

Test:
```python
r = report()
next(r)       # prime
r.send(10)
r.send(20)
r.send(30)
r.send(None)  # → "Average was: 20.00"
```

In [ ]:
# Exercise 1.3 — your code here


## Exercise 1.4 — Nested yield from (deep delegation)

Chain `yield from` three levels deep:
- `level_3()` — yields a value, receives via `.send()`, returns a result
- `level_2()` — `result = yield from level_3()`
- `level_1()` — `result = yield from level_2()`

Add prints in each level. Then `.send('hello')` on level_1 and verify it arrives at level_3.

This is how deeply nested `await` chains work — each `yield from` / `await` is transparent.

In [ ]:
# Exercise 1.4 — your code here


## Exercise 1.5 — yield from IS await (proof)

Use `@types.coroutine` to create a generator that can be used with `await`:

```python
import types

@types.coroutine
def old_sleep(seconds):
    yield ("sleep", seconds)  # yield a command to the scheduler
```

Then write an `async def new_task()` that `await`s `old_sleep(1)`.

Manually drive the coroutine with `.send(None)` — see the command it yields, then resume it.

**The big reveal:** The coroutine yielded `("sleep", 1)` — a COMMAND to the scheduler. `await asyncio.sleep(1)` does the exact same thing. The event loop is just a while-loop that receives commands from coroutines and executes them.

In [ ]:
import types

# Exercise 1.5 — your code here


## Summary

| Generator concept | Async equivalent |
|-------------------|------------------|
| `yield value` | Coroutine suspends, sends command to event loop |
| `value = yield` | Coroutine receives result from event loop |
| `yield from sub()` | `await sub()` |
| `return value` (raises StopIteration) | Coroutine returns |
| `.send(data)` | Event loop resumes coroutine with I/O result |
| `.throw(exc)` | `task.cancel()` |

**The big reveal:** `await` IS `yield from`. Python 3.5 just added nicer syntax.

**Next notebook:** Build a cooperative task scheduler using ONLY generators